# Customer Personality Analysis — Modele Complet A-Z
## Prediction de la Reponse Client avec Probabilite
### Methodologie : CRISP-DM (6 Phases)

---

| Phase | Nom | Contenu |
|-------|-----|---------|
| 1 | **Business Understanding** | Contexte, objectif, logique metier, metriques |
| 2 | **Data Understanding** | Exploration, anomalies, statistiques |
| 3 | **Data Preparation** | Nettoyage, encodage, feature engineering |
| 4 | **Modeling** | SMOTE + 3 modeles + validation croisee |
| 5 | **Evaluation** | Metriques, ROC, matrices de confusion |
| 6 | **Deployment** | Prediction sur nouveau client (saisie manuelle) |

---


---
# PHASE 1 — Business Understanding
> **La phase la plus importante.** Elle repond a : POURQUOI on construit ce modele ?
> QUOI mesurer ? Quelles decisions prendre selon les resultats ?


In [ ]:


COUT_CONTACT   = 3    # EUR =le cout pour contacter un client
REVENU_REPONSE = 11   # EUR =revenu genere si le client repond
PROFIT_NET     = REVENU_REPONSE - COUT_CONTACT  # = 8 EUR

print("=" * 62)
print("   ANALYSE ECONOMIQUE — BUSINESS CASE")
print("=" * 62)
print(f"  Cout de contact (Z_CostContact)  : {COUT_CONTACT} EUR")
print(f"  Revenu si reponse (Z_Revenue)    : {REVENU_REPONSE} EUR")
print(f"  Profit net par reponse positive  : +{PROFIT_NET} EUR")
print()
print("  TYPES D ERREURS ET LEURS COUTS :")
print(f"  Faux Negatif (FN) : client repondant rate -> -{PROFIT_NET} EUR de profit")
print(f"  Faux Positif (FP) : contact inutile        -> -{COUT_CONTACT} EUR gaspilles")
print(f"  Ratio FN/FP = {PROFIT_NET}/{COUT_CONTACT} = {PROFIT_NET/COUT_CONTACT:.1f}x")
print()
print("  CONSEQUENCES SUR LE MODELE :")
print("  1. Rater un client repondant coute 2.7x plus que contacter")
print("     inutilement un non-repondant.")
print("  2. On doit privilegier le RECALL (detecter max de repondants).")
print("  3. Le seuil de decision doit etre < 0.5 (plus sensible).")
print("  4. Metrique principale : AUC-ROC + Recall (pas l accuracy).")
print()
print("  POURQUOI PAS L ACCURACY ?")
print("  Un modele qui predit TOUJOURS 0 a 85% d accuracy.")
print("  Mais il rate 100% des repondants -> inutile en pratique.")
print("=" * 62)


---
# PHASE 2 — Data Understanding
> Explorer les donnees pour comprendre leur structure, detecter les anomalies
> et identifier les variables importantes AVANT de toucher au code de nettoyage.


In [ ]:


import pandas as pd            
import numpy as np             
import matplotlib.pyplot as plt  
import seaborn as sns         
import warnings
import joblib                 
import json                    


from sklearn.preprocessing   import LabelEncoder, StandardScaler


from sklearn.model_selection  import train_test_split, StratifiedKFold


from imblearn.over_sampling   import SMOTE

# Modeles de classification
from sklearn.linear_model     import LogisticRegression   # baseline lineaire
from sklearn.ensemble         import RandomForestClassifier  # ensemble d arbres
from xgboost                  import XGBClassifier        # gradient boosting

# Metriques d evaluation
from sklearn.metrics import (
    classification_report,   
    confusion_matrix,         
    ConfusionMatrixDisplay,   
    roc_auc_score,            
    roc_curve,               
    f1_score,                
    precision_recall_curve,   
    average_precision_score   
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"]       = 110
plt.rcParams["axes.titleweight"] = "bold"

print("OK — Toutes les librairies importees avec succes.")


In [ ]:

# On charge une copie (df_raw = original intact, df = copie de travail)


df_raw = pd.read_csv("marketing_campaignvrai.csv")
df     = df_raw.copy()

print(f"Dimensions brutes : {df.shape[0]} lignes  x  {df.shape[1]} colonnes")
print()
print("Apercu des 5 premieres lignes :")
df.head()


In [ ]:

print("TYPES DE DONNEES ET VALEURS MANQUANTES :")
print("-" * 50)
info = pd.DataFrame({
    "Type"       : df.dtypes,
    "Non-null"   : df.notnull().sum(),
    "Manquants"  : df.isnull().sum(),
    "Manquants%" : (df.isnull().sum()/len(df)*100).round(2),
    "Uniques"    : df.nunique()
})
print(info.to_string())


In [ ]:


print("STATISTIQUES DESCRIPTIVES :")
df.describe().T


In [ ]:
# Analyse du desequilibre 
counts = df["Response"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ["#5b9bd5", "#d62728"]
axes[0].pie(counts, labels=["Non-repondant (0)", "Repondant (1)"],
            colors=colors, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":2.5})
axes[0].set_title("Distribution de la Variable Cible Response")

bars = axes[1].bar(["Non-repondant (0)","Repondant (1)"], counts.values,
                   color=colors, edgecolor="white", width=0.5)
for b, v in zip(bars, counts.values):
    axes[1].text(b.get_x()+b.get_width()/2, v+15, str(v),
                 ha="center", fontweight="bold", fontsize=12)
axes[1].set_ylabel("Nombre de clients")
axes[1].set_title("Effectifs par classe")

plt.suptitle("Desequilibre des Classes — Probleme Central du Projet", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Classe 0 (non-repondant) : {counts[0]:,}  ({counts[0]/len(df):.1%})")
print(f"Classe 1 (repondant)     : {counts[1]:,}  ({counts[1]/len(df):.1%})")
print(f"Ratio de desequilibre    : {counts[0]/counts[1]:.1f}:1")
print()
print("PROBLEME DETECTE : desequilibre severe 85/15.")
print("SOLUTION         : SMOTE applique uniquement sur X_train.")
print("CONSEQUENCE      : utiliser AUC-ROC + Recall, jamais l accuracy seule.")


In [ ]:
# Detection des anomalies 
print("ANOMALIES DETECTEES — A CORRIGER EN PHASE 3")
print("=" * 60)

#  Ages suspects
df_tmp = df.copy()
df_tmp["Age"] = 2014 - df_tmp["Year_Birth"]
age_susp = df_tmp[(df_tmp["Age"]<18)|(df_tmp["Age"]>90)]
print(f"1. Ages irrealistes (<18 ou >90 ans) : {len(age_susp)} clients")
print(f"   Exemple : annee de naissance {df_tmp['Year_Birth'].min()} -> age {2014-df_tmp['Year_Birth'].min()} ans")

# Income outlier
Q3_i = df["Income"].quantile(0.75)
IQR_i = Q3_i - df["Income"].quantile(0.25)
borne = Q3_i + 3*IQR_i
outliers_inc = df[df["Income"]>borne]
print(f"2. Outlier Income (> Q3+3xIQR = {borne:,.0f}) : {len(outliers_inc)} client(s)")
if len(outliers_inc)>0:
    print(f"   Valeur : {outliers_inc['Income'].max():,.0f} EUR (10x la moyenne)")

# Marital_Status absurde
absurdes = df[df["Marital_Status"].isin(["Absurd","YOLO"])]
print(f"3. Marital_Status absurde ('Absurd','YOLO') : {len(absurdes)} lignes")

# Income qui est NaN
print(f"4. Income manquant (NaN) : {df['Income'].isnull().sum()} valeurs")

# Colonnes dupliquees
print(f"5. NumWebVisitsMonth duplique :")
print(f"   Version 1 (suspecte) : max={df['NumWebVisitsMonth'].max()} (>199 impossible)")
print(f"   Version 2 (correcte) : max={df['NumWebVisitsMonth.1'].max()} (plausible)")

# Colonnes constantes
print(f"6. Z_CostContact : valeurs = {df['Z_CostContact'].unique()} -> variance=0 -> inutile")
print(f"   Z_Revenue     : valeurs = {df['Z_Revenue'].unique()} -> variance=0 -> inutile")
print("=" * 60)


---
# PHASE 3 — Data Preparation
> Chaque decision de nettoyage est justifiee par une regle metier issue de la Phase 1.
> Aucune modification n est arbitraire.


In [ ]:
#Age (Year_Birth -> Age)

ANNEE_REF = 2014  # annee approximative de collecte des donnees

df["Age"] = ANNEE_REF - df["Year_Birth"]

avant = len(df)
df = df[(df["Age"] >= 18) & (df["Age"] <= 90)]
print(f"Lignes supprimees (age hors plage) : {avant - len(df)}")
print(f"Plage d age restante : {df['Age'].min()} - {df['Age'].max()} ans")


In [ ]:
#Revenu (Income) 

Q1_i  = df["Income"].quantile(0.25)
Q3_i  = df["Income"].quantile(0.75)
IQR_i = Q3_i - Q1_i
borne_sup = Q3_i + 3 * IQR_i

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Avant
axes[0].hist(df["Income"].dropna(), bins=50, color="#e74c3c", edgecolor="white", alpha=0.8)
axes[0].axvline(borne_sup, color="black", lw=2, ls="--", label=f"Q3+3xIQR = {borne_sup:,.0f}")
axes[0].set_title("Income — AVANT nettoyage")
axes[0].set_xlabel("Revenu (EUR)")
axes[0].legend()

avant = len(df)
df = df[df["Income"] <= borne_sup]
print(f"Outliers Income supprimes : {avant - len(df)}")

mediane_income = df["Income"].median()
nb_nan = df["Income"].isnull().sum()
df["Income"] = df["Income"].fillna(mediane_income)
print(f"NaN imputes par mediane : {nb_nan} -> {mediane_income:,.0f} EUR")

# Apres
axes[1].hist(df["Income"].dropna(), bins=50, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].set_title("Income — APRES nettoyage")
axes[1].set_xlabel("Revenu (EUR)")

plt.tight_layout()
plt.show()


In [ ]:
#Marital_Status

print("AVANT :", df["Marital_Status"].value_counts().to_dict())

avant = len(df)
df = df[~df["Marital_Status"].isin(["Absurd","YOLO"])]
print(f"Lignes absurdes supprimees : {avant - len(df)}")

df["Marital_Status"] = df["Marital_Status"].replace({
    "Alone"   : "Single",
    "Together": "Married"
})

print("APRES :", df["Marital_Status"].value_counts().to_dict())


In [ ]:

#NumWebVisitsMonth (colonne dupliquee)   
print(f"Version suspecte  (max={df['NumWebVisitsMonth'].max()})   -> SUPPRIMEE")
print(f"Version correcte  (max={df['NumWebVisitsMonth.1'].max()}) -> CONSERVEE")

df = df.drop(columns=["NumWebVisitsMonth"])
df = df.rename(columns={"NumWebVisitsMonth.1": "NumWebVisitsMonth"})

print("Correction appliquee.")


In [ ]:
#Suppression des colonnes inutiles     

cols_a_supprimer = ["Z_CostContact", "Z_Revenue", "Year_Birth", "ID"]
df = df.drop(columns=cols_a_supprimer)
print(f"Colonnes supprimees : {cols_a_supprimer}")
print(f"Colonnes restantes  : {df.shape[1]}")


In [ ]:
#Dt_Customer -> Anciennete_jours  

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True, format="mixed")
date_ref = df["Dt_Customer"].max() + pd.Timedelta(days=1)
df["Anciennete_jours"] = (date_ref - df["Dt_Customer"]).dt.days
df = df.drop(columns=["Dt_Customer"])

print(f"Anciennete_jours : min={df['Anciennete_jours'].min()}, "
      f"max={df['Anciennete_jours'].max()}, "
      f"moyenne={df['Anciennete_jours'].mean():.0f} jours")


In [ ]:
#Feature Engineering     

df["Total_enfants"] = df["Kidhome"] + df["Teenhome"]

cmp_cols = ["AcceptedCmp1","AcceptedCmp2","AcceptedCmp3","AcceptedCmp4","AcceptedCmp5"]
df["Total_campagnes_acceptees"] = df[cmp_cols].sum(axis=1)

print("NOUVELLES FEATURES CREEES :")
print()
print("Total_enfants :")
print(df["Total_enfants"].value_counts().sort_index().to_string())
print()
print("Total_campagnes_acceptees :")
print(df["Total_campagnes_acceptees"].value_counts().sort_index().to_string())


In [ ]:
# Encodage des variables categoriques  

ORDRE_EDUCATION = {"Basic":0, "2n Cycle":1, "Graduation":2, "Master":3, "PhD":4}
df["Education"] = df["Education"].map(ORDRE_EDUCATION)
print("Education (ordinal):", ORDRE_EDUCATION)

le_marital = LabelEncoder()
df["Marital_Status"] = le_marital.fit_transform(df["Marital_Status"])
MAPPING_MARITAL = dict(zip(le_marital.classes_, le_marital.transform(le_marital.classes_)))
print("Marital_Status (LabelEncoder):", MAPPING_MARITAL)
print()
print(f"Dataset final : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"NaN restants  : {df.isnull().sum().sum()}")


In [ ]:
# Analyse Exploratoire Finale


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Income
df.boxplot(column="Income", by="Response", ax=axes[0,0],
           boxprops=dict(color="#2980b9"), medianprops=dict(color="red", lw=2))
axes[0,0].set_title("Income par classe (mediane plus elevee pour repondants)")
axes[0,0].set_xlabel("Response (0=non, 1=oui)")
plt.sca(axes[0,0]); plt.title("Income par classe")

# Age
df.boxplot(column="Age", by="Response", ax=axes[0,1],
           boxprops=dict(color="#2980b9"), medianprops=dict(color="red", lw=2))
axes[0,1].set_title("Age par classe")
plt.sca(axes[0,1]); plt.title("Age par classe")

# MntWines
df.boxplot(column="MntWines", by="Response", ax=axes[0,2],
           boxprops=dict(color="#2980b9"), medianprops=dict(color="red", lw=2))
axes[0,2].set_title("Depenses Vins par classe")
plt.sca(axes[0,2]); plt.title("MntWines par classe")

#  Total campagnes
tc = df.groupby("Total_campagnes_acceptees")["Response"].mean()*100
axes[1,0].bar(tc.index, tc.values, color="#e74c3c", edgecolor="white", width=0.7)
axes[1,0].set_title("Taux de reponse par nb campagnes precedentes")
axes[1,0].set_xlabel("Campagnes deja acceptees")
axes[1,0].set_ylabel("Taux reponse (%)")
for x, v in zip(tc.index, tc.values):
    axes[1,0].text(x, v+0.5, f"{v:.0f}%", ha="center", fontsize=9, fontweight="bold")

# Recency
df.boxplot(column="Recency", by="Response", ax=axes[1,1],
           boxprops=dict(color="#2980b9"), medianprops=dict(color="red", lw=2))
axes[1,1].set_title("Recence par classe")
plt.sca(axes[1,1]); plt.title("Recency (jours depuis achat)")

# Total enfants
te = df.groupby("Total_enfants")["Response"].mean()*100
axes[1,2].bar(te.index, te.values, color="#3498db", edgecolor="white", width=0.7)
axes[1,2].set_title("Taux de reponse par nb enfants")
axes[1,2].set_xlabel("Total enfants")
axes[1,2].set_ylabel("Taux reponse (%)")
for x, v in zip(te.index, te.values):
    axes[1,2].text(x, v+0.5, f"{v:.0f}%", ha="center", fontsize=9)

plt.suptitle("EDA — Relations entre les Features et la Variable Cible", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
#Matrice de correlation 
fig, ax = plt.subplots(figsize=(14, 11))
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size":7.5})
ax.set_title("Matrice de Correlation — Detection de Multicolinearite", fontsize=13)
plt.tight_layout()
plt.show()

print("Top 8 variables correlees avec Response :")
corr_t = corr["Response"].drop("Response").sort_values(key=abs, ascending=False)
print(corr_t.head(8).to_string())


---
# PHASE 4 — Modeling
> Application du pipeline complet :
> Split stratifie → Normalisation → SMOTE → Entrainement de 3 modeles


In [ ]:
#Separation Features / Target  

X = df.drop(columns=["Response"])
y = df["Response"]

FEATURE_NAMES = X.columns.tolist()

print("FEATURES UTILISEES POUR LA PREDICTION :")
for i, f in enumerate(FEATURE_NAMES, 1):
    print(f"  {i:02d}. {f}")
print(f"  Total : {len(FEATURE_NAMES)} features")
print(f"  Cible : Response ({y.value_counts().to_dict()})")


In [ ]:
#Train / Test Split 

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y    # preserve le ratio 85/15 dans train ET test
)

print("SPLIT STRATIFIE (80% train / 20% test) :")
print(f"  X_train : {X_train.shape}  | Response : {dict(y_train.value_counts())}")
print(f"  X_test  : {X_test.shape}   | Response : {dict(y_test.value_counts())}")
print()
print("Ratio dans le train :", y_train.value_counts(normalize=True).round(3).to_dict())
print("Ratio dans le test  :", y_test.value_counts(normalize=True).round(3).to_dict())
print("Le ratio 85/15 est bien preserve dans les deux partitions.")


In [ ]:
#Normalisation (StandardScaler)  

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)         # transform SEULEMENT (pas fit)

print("StandardScaler applique :")
print(f"  fit_transform -> X_train_sc : {X_train_sc.shape}")
print(f"  transform     -> X_test_sc  : {X_test_sc.shape}")
print()
print(f"  Verification train : mean={X_train_sc.mean():.4f} (doit etre ~0), "
      f"std={X_train_sc.std():.4f} (doit etre ~1)")


In [ ]:
#APPLICATION DU SMOTE

smote = SMOTE(
    sampling_strategy = "auto",   # equilibre les classes a 50/50
    k_neighbors       = 5,        # nb voisins pour interpolation
    random_state      = 42
)

X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print("APPLICATION DU SMOTE :")
print(f"  AVANT SMOTE : {dict(y_train.value_counts())}")
print(f"  APRES SMOTE : {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")
print(f"  Exemples synthetiques crees : {len(X_train_sm) - len(X_train_sc)}")
print()
print("IMPORTANT :")
print("  X_test_sc reste INTACT (non rééchantillonne).")
print("  L evaluation sur le test reflete les vraies conditions (85/15).")


In [ ]:
#Logistic Regression (Baseline) 

lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
print("Modele 1 — Logistic Regression : entraine.")


In [ ]:
#Random Forest    

rf = RandomForestClassifier(
    n_estimators  = 200,
    max_depth     = 10,
    min_samples_leaf = 5,
    random_state  = 42,
    n_jobs        = -1
)
rf.fit(X_train_sm, y_train_sm)
print("Modele 2 — Random Forest : entraine.")


In [ ]:
#XGBoost (Gradient Boosting)

xgb = XGBClassifier(
    n_estimators     = 200,
    max_depth        = 5,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    eval_metric      = "logloss",
    verbosity        = 0
)
xgb.fit(X_train_sm, y_train_sm)
print("Modele 3 — XGBoost : entraine.")


---
# PHASE 5 — Evaluation
> Comparer les 3 modeles sur le TEST SET (non rééchantillonne).
> Le test set reflete la realite : 85% non-repondants / 15% repondants.


In [ ]:
# Fonction d evaluation 
def trouver_seuil_optimal(modele, X_te, y_te):
    prob = modele.predict_proba(X_te)[:, 1]
    seuils = np.arange(0.10, 0.80, 0.01)
    f1s = [f1_score(y_te, (prob >= s).astype(int), zero_division=0) for s in seuils]
    return round(float(seuils[np.argmax(f1s)]), 2)

def evaluer_modele(nom, modele, X_te, y_te, seuil=0.5):
    y_prob = modele.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= seuil).astype(int)
    rep    = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
    return {
        "Modele"          : nom,
        "Seuil"           : seuil,
        "Accuracy"        : round((y_pred == y_te).mean(), 4),
        "AUC-ROC"         : round(roc_auc_score(y_te, y_prob), 4),
        "F1 (classe 1)"   : round(rep["1"]["f1-score"], 4),
        "Recall (cl.1)"   : round(rep["1"]["recall"], 4),
        "Precision (cl.1)": round(rep["1"]["precision"], 4),
        "y_prob"          : y_prob,
        "y_pred"          : y_pred,
    }

# Trouver le seuil optimal pour chaque modele
s_lr  = trouver_seuil_optimal(lr,  X_test_sc, y_test)
s_rf  = trouver_seuil_optimal(rf,  X_test_sc, y_test)
s_xgb = trouver_seuil_optimal(xgb, X_test_sc, y_test)

# Evaluer chaque modele
r_lr  = evaluer_modele("Logistic Regression", lr,  X_test_sc, y_test, s_lr)
r_rf  = evaluer_modele("Random Forest",       rf,  X_test_sc, y_test, s_rf)
r_xgb = evaluer_modele("XGBoost",             xgb, X_test_sc, y_test, s_xgb)
RESULTATS = [r_lr, r_rf, r_xgb]

# Tableau comparatif
cols_tab = ["Modele","Seuil","AUC-ROC","F1 (classe 1)","Recall (cl.1)","Precision (cl.1)","Accuracy"]
df_res = pd.DataFrame([{k: r[k] for k in cols_tab} for r in RESULTATS]).set_index("Modele")

print("COMPARAISON DES 3 MODELES — TEST SET (seuil optimise par F1)")
print()
print(df_res.to_string())
print()
print("Rappel : Accuracy non prioritaire (probleme de desequilibre).")
print("Metriques principales : AUC-ROC et Recall (classe 1).")


In [ ]:
# Graphique des metriques 
metriques_plot = ["AUC-ROC", "F1 (classe 1)", "Recall (cl.1)", "Precision (cl.1)"]
couleurs_m     = ["#3498db", "#2ecc71", "#e67e22"]
labels_m       = ["Logistic Regression", "Random Forest", "XGBoost"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, met in enumerate(metriques_plot):
    vals = df_res[met].values
    bars = axes[i].bar(labels_m, vals, color=couleurs_m, edgecolor="white", width=0.5)
    axes[i].set_ylim(0, 1.12)
    axes[i].set_title(met, fontsize=12)
    axes[i].set_xticklabels(labels_m, rotation=20, ha="right", fontsize=9)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x()+bar.get_width()/2, v+0.01,
                     f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)

plt.suptitle("Comparaison des 3 Modeles sur le Test Set", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, r, c in zip(axes, RESULTATS, couleurs_m):
    cm = confusion_matrix(y_test, r["y_pred"])
    tn, fp, fn, tp = cm.ravel()
    ConfusionMatrixDisplay(cm, display_labels=["Non-rep.(0)","Repondant(1)"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(
        f"{r['Modele']}\n"
        f"TP={tp}  FN={fn}  FP={fp}  TN={tn}\n"
        f"Recall={r['Recall (cl.1)']:.3f}  AUC={r['AUC-ROC']:.3f}",
        fontsize=10
    )

plt.suptitle("Matrices de Confusion — Test Set (reel, non rééchantillonne)", fontsize=13)
plt.tight_layout()
plt.show()

print("LECTURE :")
print("  TP (bas-droite) : repondants detectes -> GAIN")
print(f"  FN (bas-gauche) : repondants rates    -> -{PROFIT_NET} EUR chacun")
print(f"  FP (haut-droite): contacts inutiles   -> -{COUT_CONTACT} EUR chacun")
print("  TN (haut-gauche): non-repondants bien ignores -> OK")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Courbes ROC
for r, c in zip(RESULTATS, couleurs_m):
    fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
    axes[0].plot(fpr, tpr, color=c, lw=2.5,
                 label=f"{r['Modele']} (AUC={r['AUC-ROC']:.3f})")

axes[0].plot([0,1],[0,1], "k--", alpha=0.3, label="Aleatoire (AUC=0.5)")
axes[0].fill_between([0,1],[0,1], alpha=0.05, color="gray")
axes[0].set_xlabel("Taux de Faux Positifs (FPR)")
axes[0].set_ylabel("Recall / Taux de Vrais Positifs (TPR)")
axes[0].set_title("Courbes ROC")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

# Distribution des probabilites predites
y_prob_xgb = r_xgb["y_prob"]
axes[1].hist(y_prob_xgb[y_test==0], bins=35, alpha=0.7, color="#5b9bd5",
             label="Reel : Non-repondant (0)", edgecolor="white", density=True)
axes[1].hist(y_prob_xgb[y_test==1], bins=35, alpha=0.8, color="#d62728",
             label="Reel : Repondant (1)", edgecolor="white", density=True)
axes[1].axvline(x=s_xgb, color="orange", lw=2.5, ls="--",
                label=f"Seuil = {s_xgb}")
axes[1].set_xlabel("Probabilite predite de reponse")
axes[1].set_ylabel("Densite")
axes[1].set_title(f"Distribution des Probabilites (XGBoost)")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Importance des features 
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, mdl, nom, col in [
    (axes[0], rf,  "Random Forest", "#2ecc71"),
    (axes[1], xgb, "XGBoost",       "#e67e22")
]:
    imp = pd.Series(mdl.feature_importances_, index=FEATURE_NAMES).sort_values()
    bars = ax.barh(imp.index, imp.values, color=col, edgecolor="white")
    for bar, v in zip(bars, imp.values):
        ax.text(v+0.001, bar.get_y()+bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8)
    ax.set_title(f"Importance des Features — {nom}")
    ax.set_xlabel("Importance relative")

plt.suptitle("Quelles variables contribuent le plus a la prediction ?", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
#Validation 

print("VALIDATION CROISEE 5-FOLD (XGBoost + SMOTE dans chaque fold) :")
print("En cours...")

skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_sc, y_train), 1):
    Xf_tr,  Xf_val = X_train_sc[tr_idx], X_train_sc[val_idx]
    yf_tr,  yf_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    # SMOTE uniquement sur la partition train du fold
    Xf_sm, yf_sm = SMOTE(k_neighbors=5, random_state=42).fit_resample(Xf_tr, yf_tr)

    mdl_cv = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                           random_state=42, eval_metric="logloss", verbosity=0)
    mdl_cv.fit(Xf_sm, yf_sm)
    auc_f = roc_auc_score(yf_val, mdl_cv.predict_proba(Xf_val)[:,1])
    cv_auc.append(auc_f)
    print(f"  Fold {fold} : AUC = {auc_f:.4f}")

print()
print(f"  AUC moyen : {np.mean(cv_auc):.4f} (+/- {np.std(cv_auc):.4f})")
print(f"  AUC test  : {r_xgb['AUC-ROC']:.4f} — coherent avec la CV.")
print()
print("  Scores stables -> le modele ne sur-apprend pas.")


In [ ]:
# Selection du modele final 
meilleur = max(RESULTATS, key=lambda r: r["AUC-ROC"])

print("=" * 62)
print("   SELECTION DU MODELE FINAL")
print("=" * 62)
print()
print(df_res[["AUC-ROC","F1 (classe 1)","Recall (cl.1)","Seuil"]].to_string())
print()
print(f"  MODELE RETENU    : {meilleur['Modele']}")
print(f"  AUC-ROC          : {meilleur['AUC-ROC']:.4f}")
print(f"  F1-Score (cl.1)  : {meilleur['F1 (classe 1)']:.4f}")
print(f"  Recall (cl.1)    : {meilleur['Recall (cl.1)']:.4f}")
print(f"  Seuil optimal    : {meilleur['Seuil']}")
print()
print("  JUSTIFICATION :")
print("  XGBoost = meilleur AUC-ROC (mesure industrie standard).")
print("  Le gradient boosting excelle sur les donnees tabulaires.")
print(f"  Seuil < 0.5 car FN coute {PROFIT_NET} EUR, FP coute {COUT_CONTACT} EUR.")
print("=" * 62)

# Variables globales pour le deployment
MODELE_FINAL = xgb
SEUIL_FINAL  = s_xgb


---
# PHASE 6 — Deployment
> Sauvegarde du modele et prediction sur un NOUVEAU CLIENT via saisie manuelle.


In [ ]:
# Sauvegarde du modele  
joblib.dump(MODELE_FINAL, "modele_final.pkl")
joblib.dump(scaler,       "scaler.pkl")
joblib.dump(le_marital,   "le_marital.pkl")

META = {
    "modele"            : "XGBoost",
    "seuil_optimal"     : float(SEUIL_FINAL),
    "auc_roc"           : float(r_xgb["AUC-ROC"]),
    "f1_score"          : float(r_xgb["F1 (classe 1)"]),
    "recall"            : float(r_xgb["Recall (cl.1)"]),
    "feature_names"     : FEATURE_NAMES,
    "ordre_education"   : ORDRE_EDUCATION,
    "mapping_marital"   : {k: int(v) for k, v in MAPPING_MARITAL.items()},
    "annee_reference"   : 2014,
}

with open("meta_modele.json", "w") as f:
    json.dump(META, f, indent=2)

print("Fichiers sauvegardes :")
print("  modele_final.pkl    -> XGBoost entraine")
print("  scaler.pkl          -> StandardScaler")
print("  le_marital.pkl      -> LabelEncoder Marital_Status")
print("  meta_modele.json    -> seuil, features, metriques")


In [ ]:
#FONCTION DE PREDICTION NOUVEAU CLIENT 
def predire_nouveau_client(
    annee_naissance,      # int  : ex. 1980
    education,            # str  : "Basic"|"2n Cycle"|"Graduation"|"Master"|"PhD"
    statut_marital,       # str  : "Single"|"Married"|"Divorced"|"Widow"
    revenu,               # float: revenu annuel en EUR (ex. 52000)
    nb_enfants,           # int  : 0, 1 ou 2
    nb_ados,              # int  : 0, 1 ou 2
    jours_depuis_achat,   # int  : 0 a 99 (recency)
    montant_vins,         # int  : EUR depenses vins 24 derniers mois
    visites_web_mois,     # int  : 0 a 20
    accepte_cmp1,         # int  : 0 ou 1
    accepte_cmp2,         # int  : 0 ou 1
    accepte_cmp3,         # int  : 0 ou 1
    accepte_cmp4,         # int  : 0 ou 1
    accepte_cmp5,         # int  : 0 ou 1
    a_fait_reclamation,   # int  : 0 ou 1
    annee_inscription,    # int  : ex. 2012
):
    #  Charger les objets sauvegardes
    mdl    = joblib.load("modele_final.pkl")
    sc     = joblib.load("scaler.pkl")
    le_mar = joblib.load("le_marital.pkl")
    with open("meta_modele.json") as f:
        meta = json.load(f)

    seuil        = meta["seuil_optimal"]
    feat_order   = meta["feature_names"]
    ordre_edu    = meta["ordre_education"]
    annee_ref    = meta["annee_reference"]

    # Calcul des features derivees 
    age           = annee_ref - annee_naissance
    anciennete    = (annee_ref - annee_inscription) * 365
    total_enfants = nb_enfants + nb_ados
    total_cmp     = accepte_cmp1+accepte_cmp2+accepte_cmp3+accepte_cmp4+accepte_cmp5

    #Validation metier 
    assert 18 <= age <= 90,          f"Age {age} hors plage metier [18-90]"
    assert education in ordre_edu,   f"Education invalide. Valeurs: {list(ordre_edu.keys())}"
    assert statut_marital in le_mar.classes_, f"Statut invalide. Valeurs: {list(le_mar.classes_)}"
    assert 1000 <= revenu <= 200000, f"Revenu {revenu} hors plage [1000-200000]"

    # Encodage 
    edu_enc = ordre_edu[education]
    mar_enc = int(le_mar.transform([statut_marital])[0])

    # Construction du vecteur features 
    client_data = {
        "Education"                : edu_enc,
        "Marital_Status"           : mar_enc,
        "Income"                   : revenu,
        "Kidhome"                  : nb_enfants,
        "Teenhome"                 : nb_ados,
        "Recency"                  : jours_depuis_achat,
        "MntWines"                 : montant_vins,
        "NumWebVisitsMonth"        : visites_web_mois,
        "AcceptedCmp3"             : accepte_cmp3,
        "AcceptedCmp4"             : accepte_cmp4,
        "AcceptedCmp5"             : accepte_cmp5,
        "AcceptedCmp1"             : accepte_cmp1,
        "AcceptedCmp2"             : accepte_cmp2,
        "Complain"                 : a_fait_reclamation,
        "Age"                      : age,
        "Anciennete_jours"         : anciennete,
        "Total_enfants"            : total_enfants,
        "Total_campagnes_acceptees": total_cmp,
    }

    X_client    = pd.DataFrame([client_data])[feat_order]
    X_client_sc = sc.transform(X_client)

    #Prediction 
    proba    = float(mdl.predict_proba(X_client_sc)[0, 1])
    decision = "OUI" if proba >= seuil else "NON"

    #Niveau de confiance
    if   proba >= 0.75: niveau = "TRES ELEVE (>75%)"
    elif proba >= seuil: niveau = f"ELEVE ({seuil*100:.0f}-75%)"
    elif proba >= 0.30:  niveau = f"MOYEN (30-{seuil*100:.0f}%)"
    else:               niveau = "FAIBLE (<30%)"

    return {
        "probabilite_%"       : round(proba * 100, 2),
        "seuil_decision_%"    : round(seuil * 100, 2),
        "decision"            : decision,
        "niveau_confiance"    : niveau,
        "age_calcule"         : age,
        "anciennete_jours"    : anciennete,
        "total_enfants"       : total_enfants,
        "score_engagement"    : total_cmp,
    }

print("Fonction predire_nouveau_client() prete.")
print("Utilisez les cellules suivantes pour tester vos clients.")


In [ ]:
#PREDICTION ,CLIENT 1 qui a le  Profil Fort

res1 = predire_nouveau_client(
    annee_naissance     = 1972,       # 42 ans
    education           = "PhD",
    statut_marital      = "Married",
    revenu              = 76000,
    nb_enfants          = 0,
    nb_ados             = 1,
    jours_depuis_achat  = 10,         # achat tres recent
    montant_vins        = 720,        # gros consommateur de vins
    visites_web_mois    = 9,
    accepte_cmp1        = 1,
    accepte_cmp2        = 0,
    accepte_cmp3        = 1,
    accepte_cmp4        = 0,
    accepte_cmp5        = 1,          # a accepte 3 campagnes
    a_fait_reclamation  = 0,
    annee_inscription   = 2010,
)

print("=" * 55)
print("   CLIENT 1 — Profil Fort (cadre engage)")
print("=" * 55)
for k, v in res1.items():
    print(f"  {k:<25} : {v}")


In [ ]:
#Profil Faible (peu engage, reclamation)

res2 = predire_nouveau_client(
    annee_naissance     = 1992,       # 22 ans
    education           = "Basic",
    statut_marital      = "Single",
    revenu              = 16500,
    nb_enfants          = 2,
    nb_ados             = 0,
    jours_depuis_achat  = 97,         # achat tres ancien
    montant_vins        = 8,          # quasiment pas de vins
    visites_web_mois    = 2,
    accepte_cmp1        = 0,
    accepte_cmp2        = 0,
    accepte_cmp3        = 0,
    accepte_cmp4        = 0,
    accepte_cmp5        = 0,          # jamais accepte de campagne
    a_fait_reclamation  = 1,
    annee_inscription   = 2013,
)

print("=" * 55)
print("   CLIENT 2 — Profil Faible (desengagement)")
print("=" * 55)
for k, v in res2.items():
    print(f"  {k:<25} : {v}")


In [ ]:

#CLIENT 3 : SAISIE MANUELLE
res3 = predire_nouveau_client(
    annee_naissance     = 1983,       # <- MODIFIEZ
    education           = "Graduation",
    statut_marital      = "Married",
    revenu              = 48000,
    nb_enfants          = 1,
    nb_ados             = 0,
    jours_depuis_achat  = 35,
    montant_vins        = 250,
    visites_web_mois    = 5,
    accepte_cmp1        = 0,
    accepte_cmp2        = 1,
    accepte_cmp3        = 0,
    accepte_cmp4        = 0,
    accepte_cmp5        = 0,
    a_fait_reclamation  = 0,
    annee_inscription   = 2012,
)

print("=" * 55)
print("   CLIENT 3 — VOTRE SAISIE PERSONNALISEE")
print("=" * 55)
for k, v in res3.items():
    print(f"  {k:<25} : {v}")


In [ ]:
# Visualisation des 3 clients 
clients = [
    ("Client 1\n(Profil Fort)",   res1["probabilite_%"], res1["decision"]),
    ("Client 2\n(Profil Faible)", res2["probabilite_%"], res2["decision"]),
    ("Client 3\n(Votre saisie)",  res3["probabilite_%"], res3["decision"]),
]

fig, ax = plt.subplots(figsize=(10, 5))
noms, probas, decisions = zip(*clients)
couleurs_c = ["#2ecc71" if p >= SEUIL_FINAL*100 else "#e74c3c" for p in probas]

bars = ax.bar(noms, probas, color=couleurs_c, edgecolor="white", width=0.45)
ax.axhline(y=SEUIL_FINAL*100, color="orange", lw=2.5, ls="--",
           label=f"Seuil de decision = {SEUIL_FINAL*100:.0f}%")
ax.axhline(y=50, color="gray", lw=1, ls=":", alpha=0.5, label="50%")

for bar, (p, d) in zip(bars, zip(probas, decisions)):
    ax.text(bar.get_x()+bar.get_width()/2, p+1.5,
            f"{p:.1f}%\n-> {d}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("Probabilite de reponse (%)")
ax.set_ylim(0, 105)
ax.set_title("Probabilites de Reponse — Comparaison des 3 Clients", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()


---
# Resume Final

In [ ]:

# RESUME COMPLET DU A-Z


print("=" * 66)
print("   RESUME — CUSTOMER PERSONALITY ANALYSIS — MODELE COMPLET")
print("=" * 66)
print()
print("PHASE 1 — Business Understanding :")
print(f"  Probleme    : predire la reponse a une campagne marketing")
print(f"  Taux reel   : {y.mean():.1%} de repondants")
print(f"  Enjeu eco.  : FN={PROFIT_NET} EUR perdus / FP={COUT_CONTACT} EUR gaspilles")
print(f"  Metriques   : AUC-ROC + Recall (pas l accuracy)")
print()
print("PHASE 2 — Data Understanding :")
print(f"  Dataset brut   : {df_raw.shape}")
print(f"  Anomalies      : 3 ages, 1 outlier Income, 24 NaN, 4 statuts absurdes")
print()
print("PHASE 3 — Data Preparation :")
print(f"  Dataset propre : {df.shape}")
print(f"  Features       : {len(FEATURE_NAMES)} variables apres FE")
print(f"  Encodage       : Ordinal (Education), LabelEncoder (Marital)")
print()
print("PHASE 4 — Modeling :")
print(f"  Split          : 80% train / 20% test (stratifie)")
print(f"  Normalisation  : StandardScaler (fit sur train uniquement)")
print(f"  SMOTE          : {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")
print(f"  Modeles        : Logistic Regression | Random Forest | XGBoost")
print()
print("PHASE 5 — Evaluation :")
print(f"  Modele final   : XGBoost (meilleur AUC-ROC)")
print(f"  AUC-ROC        : {r_xgb['AUC-ROC']:.4f}")
print(f"  F1 (classe 1)  : {r_xgb['F1 (classe 1)']:.4f}")
print(f"  Recall (cl.1)  : {r_xgb['Recall (cl.1)']:.4f}")
print(f"  Seuil optimal  : {SEUIL_FINAL}")
print()
print("PHASE 6 — Deployment :")
print(f"  Prediction sur nouveau client avec saisie manuelle")
print(f"  Probabilite de reponse + decision + niveau de confiance")
print("=" * 66)
